# tp-04 前段モデル（位置取り・ペース・上がり・時計の予測）と、後段への寄与

**実行場所: ホスト PC（stg）。** 先に tp-01〜tp-03 を通すこと（`features/train_table.parquet`、`ablation_nospeed/adopted_features.json`）。

## 目的
特徴量戦略書 §4・§8 の前段予測（P1 位置取り → P2 隊列の集計 → P3 ペース → P4 上がり 3F → P5 走破タイム → P6 正規化着順）を、**時系列の OOF** で作り、
(1) 各段の予測精度、(2) 後段（p_win）への寄与を確かめる。結果は `data/local/modeling/training_pipeline/oof/` に保存し、tp-05 が特徴量（`s_*`・`m_*`）として使う。

## 条件
- 入力は tp-03 で採用した特徴量（B0′ + 履歴特徴量。`speed_*` なし）だけ。目的変数はそのレースの結果で、学習のラベルにだけ使う。
- OOF: 年 Y の予測は Y の開始 10 日前（purge 7 日 + embargo 3 日）より前のデータで学習したモデルで出す（expanding）。2020〜2021 年の行には予測が付かない。2025 年（test）の行は 2020〜2024 年で学習したモデルで予測するが、**採否の判断には使わない**（比較は valid 2022〜2024 だけ）。
- 前段の OOF が使えるのは 2022 年以降なので、連鎖（P2 以降）の比較は valid 2023・2024（学習は 2022 年以降）で行う。
- 2026 年は使わない。ホストの 6 コアは他ジョブ用に空ける。

## 「どちらも試して良い方を採用」する比較
| 段 | 比較 |
|---|---|
| P1 | 入力に騎手・調教師の先行傾向（as-of）を足す／足さない。expanding と leave-one-year-out（上限の目安） |
| P3 | ラベルを pace_bal／rpci、入力に P2 の集計を使う／使わない（学習を 2022 年以降に揃えた公平な比較を含む） |
| P4 | 上がりをペース別に条件付ける／しない、P1 の予測を入力に使う／使わない |
| P5・P6 | 前段の予測を入力に使う／使わない |
| 後段 | s_*・m_* を p_win のモデルに足す／足さない（ファミリー別、F2 の簡易版との置き換え） |

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "6")   # polars を import する前に決める（ホストの余裕を確保）
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.project_env import load_project_dotenv

load_project_dotenv()
print("REPO_ROOT:", REPO_ROOT, "| KEIBA_ENV:", os.environ.get("KEIBA_ENV"))

In [ ]:
import hashlib, json, pickle, subprocess, time, gc
import numpy as np
import pandas as pd
import polars as pl
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ablation as ab
from src.pipeline.models import stage_models as sm
from src.pipeline.models.compute import detect_compute
from src.pipeline.models.time_split import yearly_expanding_folds

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
FEAT = WORK / "features"
OOF = WORK / "oof"
CACHE = OOF / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
CACHE_TAG = "v2"   # 手順を変えたらここを変える（古い途中結果を使わない）
COMPUTE = detect_compute(reserve_cores=6)   # ホストの 24 コアのうち 6 を他ジョブ（スクレイピング・cron）用に空ける
THREADS = COMPUTE.threads

MLFLOW_URI = require_tracking_server()
mlflow.set_experiment("keiba-tp-stage")
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
print("MLflow:", MLFLOW_URI, "| git:", GIT_COMMIT, "| threads:", THREADS)


def cached(name, fn):
    """途中結果を pickle に保存して、再実行では読み込む。"""
    p = CACHE / f"{CACHE_TAG}_{name}.pkl"
    if p.exists():
        return pickle.loads(p.read_bytes())
    t0 = time.time()
    out = fn()
    p.write_bytes(pickle.dumps(out))
    print(f"  [{name}] {time.time() - t0:.0f}s")
    return out

## 0. データとラベル

In [ ]:
adopted = json.loads((WORK / "ablation_nospeed/adopted_features.json").read_text(encoding="utf-8"))
BASE_FEATS = list(adopted["b0_features"]) + list(adopted["features"])
df = ab.load_table(FEAT / "train_table.parquet")
df["race_id"] = df["race_id"].astype(str)
df["horse_id"] = df["horse_id"].astype(str)
miss = [c for c in BASE_FEATS if c not in df.columns]
assert not miss, miss
assert not any(c.startswith("speed_") or c == "has_speed_index" for c in BASE_FEATS), "speed_* は入れない（運用で取得できない可能性）"
print(f"行 {len(df):,} / レース {df['race_id'].nunique():,} / 入力 {len(BASE_FEATS)} 列 / {df['date'].min():%Y-%m-%d}〜{df['date'].max():%Y-%m-%d}")
assert int(df["year"].max()) == 2025, "2026 年が混ざっています"

In [ ]:
lab_path = OOF / "stage_labels.parquet"
if lab_path.exists():
    labels = pd.read_parquet(lab_path)
else:
    on_time = pl.read_parquet(FEAT / "on_time_repair.parquet")
    labels = sm.stage_labels(WORK / "layer_a.parquet", on_time)
    labels.to_parquet(lab_path, index=False)
fit_params = json.loads((FEAT / "fit_params.json").read_text(encoding="utf-8"))
race_cls = sm.pace_class_labels(labels, fit_params["pace_q"], fit_params["rpci_q"])
T = sm.build_targets(df, labels, race_cls)
cov = T.assign(year=df["year"].values).groupby("year")[["t_pos_early", "t_pos_late", "t_l3f_dev", "t_time_dev", "t_rank_norm", "r_pace_bal_cls", "r_rpci_cls"]].apply(lambda g: g.notna().mean())
print("ラベルの有効率（年別）"); display(cov.round(3))
print("ペース区分の割合 pace_bal(0=後傾,2=前傾) / rpci(0=前傾,2=後傾):")
rc = race_cls.dropna()
print(rc["pace_bal_cls"].value_counts(normalize=True).sort_index().round(3).to_dict(), rc["rpci_cls"].value_counts(normalize=True).sort_index().round(3).to_dict())

In [ ]:
VALID_EXP = [2022, 2023, 2024, 2025]     # 2025 は test。予測は作るが、採否の比較には使わない
CMP_YEARS = (2023, 2024)                 # 連鎖の比較に使う年
RACE = df["race_id"].to_numpy()
YEAR = df["year"].to_numpy()


def score(rep):
    return {"MAE": rep["MAE"].mean(), "定数MAE": rep["定数予測のMAE"].mean(), "相関": rep["相関"].mean(), "Spearman": rep["レース内Spearman"].mean()}


def align(preds):
    """比較する予測が全部そろっている行だけを残す（行の違いで優劣がついたように見えるのを防ぐ）。"""
    ok = np.ones(len(df), bool)
    for p in preds.values():
        ok &= ~np.isnan(p)
    return {k: np.where(ok, p, np.nan) for k, p in preds.items()}


def compare(target, preds, years=CMP_YEARS):
    preds = align(preds)
    rows = {k: score(sm.reg_report(df, target, p, years)) for k, p in preds.items()}
    return pd.DataFrame(rows).T

## 1. P1 位置取り（初角・最終角の順位の正規化。0=先頭〜1=最後尾）
入力は as-of の特徴量。A: 特徴量のみ、B: 騎手・調教師の「先行傾向」（過去に騎乗・管理した馬の位置取りの平均。当日より前だけ）を足す。

In [ ]:
def _jt():
    parts = []
    for tgt in ("t_pos_early", "t_pos_late"):
        for g, gn in (("jockey_id", "jk"), ("trainer_id", "tr")):
            tmp = df[["date", g]].assign(v=T[tgt].to_numpy())
            parts.append(sm.asof_group_mean(tmp, g, "v", name=f"{gn}_{tgt[2:]}"))
    return pd.concat(parts, axis=1)


JT = cached("jt", _jt)
X_A = df[BASE_FEATS]
X_B = pd.concat([df[BASE_FEATS], JT], axis=1)
P1 = {}
for tname in ("t_pos_early", "t_pos_late"):
    y = T[tname].to_numpy()
    for vname, X in (("A", X_A), ("B", X_B)):
        P1[(tname, vname)] = cached(f"p1_{tname}_{vname}", lambda: sm.oof_regression(df, X, y, VALID_EXP, "expanding", THREADS)["base"])
for tname in ("t_pos_early", "t_pos_late"):
    y = T[tname].to_numpy()
    ref = {"参考: a_style そのまま": df["a_style"].to_numpy() if tname == "t_pos_early" else df["a_style_late"].to_numpy()}
    rows = {k: score(sm.reg_report(df, y, p, (2022, 2023, 2024))) for k, p in {"A": P1[(tname, "A")], "B": P1[(tname, "B")]}.items()}
    r0 = sm.reg_report(df, y, list(ref.values())[0], (2022, 2023, 2024))
    rows["参考: a_style(過去走の脚質)"] = {"MAE": np.nan, "定数MAE": r0["定数予測のMAE"].mean(), "相関": r0["相関"].mean(), "Spearman": r0["レース内Spearman"].mean()}
    print(tname, "(valid 2022〜2024 の平均)"); display(pd.DataFrame(rows).T.round(4))

In [ ]:
P1_CHOICE = {}
for tname in ("t_pos_early", "t_pos_late"):
    y = T[tname].to_numpy()
    sA = score(sm.reg_report(df, y, P1[(tname, "A")], (2022, 2023, 2024)))["MAE"]
    sB = score(sm.reg_report(df, y, P1[(tname, "B")], (2022, 2023, 2024)))["MAE"]
    P1_CHOICE[tname] = "B" if sB < sA else "A"
print("P1 の採用（MAE が小さい方）:", P1_CHOICE)

# leave-one-year-out（上限の目安。将来のデータで学習する先読みを含むので採用はしない）と比べる
LOYO_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
P1_LOYO = {}
for tname in ("t_pos_early", "t_pos_late"):
    X = X_B if P1_CHOICE[tname] == "B" else X_A
    y = T[tname].to_numpy()
    P1_LOYO[tname] = cached(f"p1_loyo_{tname}", lambda: sm.oof_regression(df, X, y, LOYO_YEARS, "loyo", THREADS)["base"])
    cmp_ = compare(y, {"expanding": P1[(tname, P1_CHOICE[tname])], "leave-one-year-out": P1_LOYO[tname]}, years=(2022, 2023, 2024))
    print(tname, "(同じ行で比較)"); display(cmp_.round(4))

## 2. P2 隊列の集計（P1 の予測だけから作る。モデルなし）
予測先行馬頭数（自馬を除く、予測位置が 0.25 未満の馬の数）、自馬と他馬の最前との差、単騎逃げの確率の目安。自馬の結果は使わない。

In [ ]:
s_early = P1[("t_pos_early", P1_CHOICE["t_pos_early"])]
s_late = P1[("t_pos_late", P1_CHOICE["t_pos_late"])]
P2 = sm.p2_aggregates(RACE, s_early)
S1 = P2.copy()
S1["s_pos_late_pred"] = s_late.astype("float32")
STAGE_P1 = ["s_pos_pred", "s_pos_late_pred", "s_pos_pred_rank", "s_nfront_pred", "s_front_gap", "s_lone_front_prob"]
S1 = S1[STAGE_P1]
H1 = hashlib.md5(pd.util.hash_pandas_object(S1.iloc[::40], index=False).to_numpy().tobytes()).hexdigest()[:8]   # P1・P2 の中身（途中結果の名前に入れる）
m = (YEAR >= 2022) & (YEAR <= 2024)
t_ = T["t_pos_early"].to_numpy()
print("P2 の集計と実際の初角位置:")
tmp = pd.DataFrame({"s_nfront_pred": S1["s_nfront_pred"], "実際の先行馬数": pd.Series((t_ < sm.FRONT_THR).astype(float)).groupby(RACE).transform("sum") - (t_ < sm.FRONT_THR)})
print("  s_nfront_pred と実際の先行馬数(自馬除く)の相関 (valid 2022〜2024):", round(float(tmp[m].corr().iloc[0, 1]), 3))
print("  f2_nfront(簡易版)との相関:", round(float(pd.concat([S1["s_nfront_pred"], df["f2_nfront"]], axis=1)[m].corr().iloc[0, 1]), 3))
print("  実際の先行馬数との相関 f2_nfront:", round(float(pd.concat([df["f2_nfront"], tmp["実際の先行馬数"]], axis=1)[m].corr().iloc[0, 1]), 3))

## 3. P3 ペース（前傾／平均／後傾の確率）
レース単位。ラベルは pace_bal の 3 区分（0=後傾,2=前傾）と rpci の 3 区分（0=前傾,2=後傾）の両方（閾値は tp-02 の `fit_params.json`、FIT_END 以前で決定）。
入力はレース条件 + 出走馬の as-of 特徴量の集計。P2 の集計を足す案は、学習が 2022 年以降に限られるため、同じ条件（2022 年以降）の P2 なし版と比べる。
基準は一様予測（ln 3 ≈ 1.099）と、学習期間のクラス頻度。

In [ ]:
RACE_COLS = ["distance", "field_size", "surface", "venue_code", "track_condition", "grade", "race_class", "course_type", "weight_rule",
             "direction", "round", "start_hour", "track_bias_winner_3f_same_day_prior", "track_bias_winner_3f_prev_day",
             "track_bias_winner_3f_diff", "track_bias_winner_3f_weighted"]
AGG_COLS = ["f3_r", "f3_E", "a_style", "a_style_late", "a_l3fz_ewm2", "sr_perf_ewm2", "pc_pci_ewm2", "pc_rpci_ewm3", "f2_nfront", "f2_pressure"]
P2_AGG = ["s_pos_pred", "s_nfront_pred", "s_front_gap", "s_lone_front_prob"]
for c in RACE_COLS + AGG_COLS:
    assert c in df.columns, c
R0 = sm.race_level(df, RACE_COLS, AGG_COLS)
R1 = sm.race_level(df, RACE_COLS, AGG_COLS, extra=S1, extra_aggs=P2_AGG)
R_year = R0["year"].to_numpy()
R_date = R0["date"].to_numpy()
rt = T.groupby(df["race_id"], sort=False)[["r_pace_bal", "r_rpci", "r_pace_bal_cls", "r_rpci_cls"]].first().reindex(R0.index)
X_r0 = R0.drop(columns=["date", "year"])
X_r1 = R1.drop(columns=["date", "year"])
print(f"レース {len(R0):,} / 入力列 rule {X_r0.shape[1]} / +P2 {X_r1.shape[1]}")

In [ ]:
SEEDS3 = (0, 1, 2)
LABELS = {"pace_bal": "r_pace_bal_cls", "rpci": "r_rpci_cls"}
X_course = R0[RACE_COLS + ["n_runners"]]   # 出走馬の情報を使わない（コース・距離・馬場・頭数だけ）
VARIANTS = {"コースのみ(2020〜)": (X_course, None), "rule(2020〜)": (X_r0, None), "rule(2022〜)": (X_r0, 2022), "rule+P2(2022〜)": (X_r1, 2022)}


def run_p3(lname, vname):
    Xr, train_from = VARIANTS[vname]
    y_full = rt[LABELS[lname]].to_numpy(dtype=float)
    y = y_full.copy()
    if train_from:
        y[R_year < train_from] = np.nan
    return [sm.oof_multiclass(Xr, y, R_year, R_date, VALID_EXP, "expanding", THREADS, seed=s) for s in SEEDS3]


P3 = {}
rows = []
for lname in LABELS:
    y_full = rt[LABELS[lname]].to_numpy(dtype=float)
    for vname in VARIANTS:
        Ps = cached(f"p3_{H1}_{lname}_{vname}", lambda: run_p3(lname, vname))
        P3[(lname, vname)] = Ps
        m_ = np.isin(R_year, CMP_YEARS)
        lls = [sm.multiclass_logloss(np.where(m_[:, None], P, np.nan), y_full) for P in Ps]
        Pm = np.mean(Ps, axis=0)
        prior = []
        for yv in CMP_YEARS:
            tr = y_full[(R_year < yv) & ~np.isnan(y_full)]
            pr = np.bincount(tr.astype(int), minlength=3) / len(tr)
            vm = (R_year == yv) & ~np.isnan(y_full)
            prior.append(-np.log(np.clip(pr[y_full[vm].astype(int)], 1e-12, 1)).mean())
        acc = float(np.nanmean((Pm.argmax(1) == y_full)[m_ & ~np.isnan(y_full) & ~np.isnan(Pm).any(1)]))
        rows.append({"ラベル": lname, "入力": vname, "LogLoss(seed平均)": np.mean(lls), "seed間sd": np.std(lls, ddof=1), "クラス頻度": np.mean(prior),
                     "ln3": np.log(3), "正解率": acc})
P3T = pd.DataFrame(rows)
display(P3T.round(4))
print("コースのみ → 出走馬の集計を足したときの LogLoss の改善（正なら出走馬の情報が効いている）:")
for l in LABELS:
    t = P3T[P3T["ラベル"] == l].set_index("入力")["LogLoss(seed平均)"]
    print(" ", l, round(float(t["コースのみ(2020〜)"] - t["rule(2020〜)"]), 4))

In [ ]:
# ラベルごとに、LogLoss が最も小さい入力を採用。一様予測（ln3）を seed 間のばらつきの 2 倍以上下回らなければ、使わない（F2 の簡易版に戻す）
P3_CHOICE, P3_OK = {}, {}
for lname in LABELS:
    t = P3T[P3T["ラベル"] == lname].set_index("入力")
    best = t["LogLoss(seed平均)"].idxmin()
    # 同等（best + 2×seed sd 以内）なら、より単純な入力（VARIANTS の先頭側）を採る
    tol = t.loc[best, "LogLoss(seed平均)"] + 2 * t.loc[best, "seed間sd"]
    best = next(v for v in VARIANTS if v in t.index and t.loc[v, "LogLoss(seed平均)"] <= tol)
    P3_CHOICE[lname] = best
    P3_OK[lname] = bool(np.log(3) - t.loc[best, "LogLoss(seed平均)"] > 2 * t.loc[best, "seed間sd"])
print("P3 採用:", P3_CHOICE, "| ln3 を明確に下回る:", P3_OK)
# P3 の確率（採用した入力、seed 平均）を馬に割り当てる
RIDX = R0.index.get_indexer(df["race_id"])
P3P = {l: np.mean(P3[(l, P3_CHOICE[l])], axis=0)[RIDX] for l in LABELS}
# P4 の条件付けに使うラベル: クラス頻度との差が大きい方
rel = {l: float(P3T[(P3T["ラベル"] == l) & (P3T["入力"] == P3_CHOICE[l])]["クラス頻度"].iloc[0] - P3T[(P3T["ラベル"] == l) & (P3T["入力"] == P3_CHOICE[l])]["LogLoss(seed平均)"].iloc[0]) for l in LABELS}
PACE_LABEL = max(rel, key=rel.get)
print("クラス頻度に対する改善:", {k: round(v, 4) for k, v in rel.items()}, "→ P4 の条件付けに使うラベル:", PACE_LABEL)

In [ ]:
# 期待ペース（連続値）: pace_bal と rpci を回帰で予測（レース単位）。採用した入力と同じものを使う
def _p3reg(lname):
    Xr, train_from = VARIANTS[P3_CHOICE[lname]]
    y = rt["r_pace_bal" if lname == "pace_bal" else "r_rpci"].to_numpy(dtype=float).copy()
    if train_from:
        y[R_year < train_from] = np.nan
    Rf = pd.DataFrame({"year": R_year, "date": R_date})
    return sm.oof_regression(Rf, Xr, y, VALID_EXP, "expanding", THREADS, n_rounds=200, params={"num_leaves": 15, "min_data_in_leaf": 40})["base"]


PREG = {l: cached(f"p3reg_{H1}_{P3_CHOICE[l]}_{l}", lambda: _p3reg(l)) for l in LABELS}
for l, col in (("pace_bal", "r_pace_bal"), ("rpci", "r_rpci")):
    y = rt[col].to_numpy(dtype=float)
    m_ = np.isin(R_year, CMP_YEARS) & ~np.isnan(y) & ~np.isnan(PREG[l])
    print(f"{l} 回帰: 相関 {np.corrcoef(y[m_], PREG[l][m_])[0, 1]:.3f} / MAE {np.abs(y[m_] - PREG[l][m_]).mean():.3f} / 定数MAE {np.abs(y[m_] - np.nanmean(y[R_year < 2023])).mean():.3f}")

## 4. P4 上がり 3F（レース内平均との差。秒）
A: ペースを条件にしない／B: ペースを条件に入れて、予測時に前傾・平均・後傾を仮定して P3 の確率で重み付け（Σk P3k × 予測k）。
それぞれ、P1・P2 の予測を入力に使う／使わない（使うと学習は 2022 年以降）。障害レースは除く。
「実際のペースを入れた場合」は上限の目安（結果由来なので採用しない）。

In [ ]:
PC = T[f"r_{PACE_LABEL}_cls"].to_numpy(dtype=float)
P3K = P3P[PACE_LABEL]
y4 = T["t_l3f_dev"].to_numpy()
has_s1 = ~np.isnan(S1["s_pos_pred"].to_numpy())


def run_p4(stage, cond):
    X = df[BASE_FEATS]
    y = y4.copy()
    if stage:
        X = pd.concat([X, S1[STAGE_P1]], axis=1)
        y[~has_s1] = np.nan
    variants = None
    if cond:
        X = X.assign(pace_cls=PC)
        variants = {f"k{k}": {"pace_cls": np.full(len(df), float(k))} for k in range(3)}
        variants["actual"] = {}
    return sm.oof_regression(df, X, y, VALID_EXP, "expanding", THREADS, variants=variants)


P4 = {(s, c): cached(f"p4_{H1}_{PACE_LABEL}_s{int(s)}_c{int(c)}", lambda: run_p4(s, c)) for s in (False, True) for c in (False, True)}
cand = {}
for s in (False, True):
    nm = "P1入力あり" if s else "P1入力なし"
    cand[f"{nm}/条件なし"] = P4[(s, False)]["base"]
    k = P4[(s, True)]
    cand[f"{nm}/ペース条件(P3で重み付け)"] = sum(P3K[:, i] * k[f"k{i}"] for i in range(3))
    cand[f"[上限]{nm}/実際のペース"] = k["actual"]
cmp4 = compare(y4, cand)
display(cmp4.round(4))

In [ ]:
usable = {k: v for k, v in cand.items() if not k.startswith("[上限]")}
cmp4u = compare(y4, usable)
P4_UNCOND = cmp4u[cmp4u.index.str.endswith("条件なし")]["MAE"].idxmin()
P4_COND = cmp4u[cmp4u.index.str.endswith("(P3で重み付け)")]["MAE"].idxmin()
print("P4 採用  条件なし:", P4_UNCOND, "| ペース条件:", P4_COND)
s_l3f = cand[P4_UNCOND]
s_l3f_cond = cand[P4_COND]

## 5. P5 走破タイム（レース内平均との差。秒）と P6 正規化着順
A: 前段の予測を使わない（学習は 2020 年から）／B: P1・P3・P4 の予測を入力に足す（学習は 2022 年以降）。

In [ ]:
S3 = pd.DataFrame({"s_pace_p_front": P3P["pace_bal"][:, 2], "s_pace_p_back": P3P["pace_bal"][:, 0],
                   "s_pace_bal_pred": PREG["pace_bal"][RIDX], "s_rpci_pred": PREG["rpci"][RIDX],
                   "s_l3f_pred": s_l3f, "s_l3f_pred_cond": s_l3f_cond}, index=df.index).astype("float32")
STAGE_CHAIN = STAGE_P1 + list(S3.columns)
CH = hashlib.md5(pd.util.hash_pandas_object(pd.concat([S1, S3], axis=1).iloc[::40], index=False).to_numpy().tobytes()).hexdigest()[:8]   # 前段(P1〜P4)の中身
chain_ok = ~np.isnan(S1["s_pos_pred"].to_numpy()) & ~np.isnan(S3["s_pace_p_front"].to_numpy()) & ~np.isnan(S3["s_l3f_pred_cond"].to_numpy())


def run_p56(tname, stage, extra=None):
    X = df[BASE_FEATS]
    y = T[tname].to_numpy().copy()
    if stage:
        X = pd.concat([X, S1[STAGE_P1], S3] + ([extra] if extra is not None else []), axis=1)
        y[~chain_ok] = np.nan
    return sm.oof_regression(df, X, y, VALID_EXP, "expanding", THREADS)["base"]


P5 = {s: cached(f"p5_{CH}_s{int(s)}", lambda: run_p56("t_time_dev", s)) for s in (False, True)}
y5 = T["t_time_dev"].to_numpy()
cmp5 = compare(y5, {"A:前段なし": P5[False], "B:前段あり": P5[True]})
print("P5 走破タイム差"); display(cmp5.round(4))
P5_CHOICE = cmp5["Spearman"].idxmax()
s_time = P5[True] if P5_CHOICE.startswith("B") else P5[False]

In [ ]:
S5 = pd.DataFrame({"s_time_pred": s_time}, index=df.index).astype("float32")
P6 = {s: cached(f"p6_{CH}_s{int(s)}", lambda: run_p56("t_rank_norm", s, extra=S5)) for s in (False, True)}
y6 = T["t_rank_norm"].to_numpy()
cmp6 = compare(y6, {"A:前段なし": P6[False], "B:前段あり": P6[True]})
print("P6 正規化着順"); display(cmp6.round(4))
P6_CHOICE = cmp6["Spearman"].idxmax()
s_rank = P6[True] if P6_CHOICE.startswith("B") else P6[False]
print("採用 P5:", P5_CHOICE, "| P6:", P6_CHOICE)

## 6. 後段に渡す特徴量（第 2 段）の組み立てと保存
`s_*`（予測そのもの）と `m_*`（馬の区分別成績 × 予測ペースの確率）。2020〜2021 年の行と、採用した入力が 2022 年以降に限られる段の 2022 年の行は欠損のまま。

In [ ]:
def race_rel(v, fn):
    return pd.Series(v, index=df.index).groupby(df["race_id"]).transform(fn).to_numpy()


FEATS = S1[STAGE_P1].copy()
pb, pr = P3P["pace_bal"], P3P["rpci"]
FEATS["s_pace_p_front"] = pb[:, 2]      # pace_bal: 2=前傾
FEATS["s_pace_p_mid"] = pb[:, 1]
FEATS["s_pace_p_back"] = pb[:, 0]
FEATS["s_pace_bal_pred"] = PREG["pace_bal"][RIDX]
FEATS["s_rpci_pred"] = PREG["rpci"][RIDX]
FEATS["m_pace_match"] = sum(pb[:, k] * df[f"f2_a_reg{k}"].to_numpy() for k in range(3))
FEATS["m_rpci_match"] = sum(pr[:, k] * df[f"pc_a_reg{k}"].to_numpy() for k in range(3))
# f2_pace_adv = f2_a_reg0(後傾) − f2_a_reg2(前傾)。前傾に強い馬が前傾と予測されるレースで有利、になるよう符号をそろえる
FEATS["m_pace_adv"] = -df["f2_pace_adv"].to_numpy() * (pb[:, 2] - pb[:, 0])
FEATS["s_l3f_pred"] = s_l3f
FEATS["s_l3f_pred_cond"] = s_l3f_cond
FEATS["s_l3f_adv"] = s_l3f_cond - race_rel(s_l3f_cond, "mean")
FEATS["s_time_pred"] = s_time
FEATS["s_time_gap"] = s_time - race_rel(s_time, "min")
FEATS["s_rank_pred"] = s_rank
FEATS = FEATS.astype("float32")
for l in LABELS:
    if not P3_OK[l]:
        cols = [c for c in FEATS.columns if (l == "pace_bal" and c in ("s_pace_p_front", "s_pace_p_mid", "s_pace_p_back", "s_pace_bal_pred", "m_pace_match", "m_pace_adv"))
                or (l == "rpci" and c in ("s_rpci_pred", "m_rpci_match"))]
        FEATS = FEATS.drop(columns=cols)
        print(f"P3({l}) は ln3 を明確に下回らないため、{cols} は渡さない（F2 の簡易版に戻す）")
S_ALL = list(FEATS.columns)
print(len(S_ALL), "列:", S_ALL)
print(FEATS.groupby(YEAR).apply(lambda g: g.notna().mean().mean()).round(3).to_dict(), "← 年別の非欠損率の平均")
out = pd.concat([df[["race_id", "horse_id"]], FEATS], axis=1)
out["loyo_s_pos_pred"] = P1_LOYO["t_pos_early"].astype("float32")
out.to_parquet(OOF / "stage_features_expanding.parquet", index=False)
print("保存:", OOF / "stage_features_expanding.parquet", out.shape)

## 7. 後段への寄与（p_win の LogLoss）
tp-03 と同じ手順（LightGBM 固定・3 seed・purge+embargo）。前段の特徴量がある行は 2022 年以降なので、**学習を 2022 年以降に揃えた base** と比べ、valid は 2023・2024。
採否は tp-03 の基準（全 fold で悪化せず、平均が max(0.0001, 2×seed sd) 以上改善）。fold が 2 つだけなので、確定ではなく傾向として扱い、tp-05 で再確認する。

In [ ]:
for c in S_ALL:
    df[c] = FEATS[c].to_numpy()
folds = list(yearly_expanding_folds(df))
FAM_S = {
    "P1(位置取り・隊列)": [c for c in S_ALL if c in STAGE_P1],
    "P3(ペース確率・期待ペース)": [c for c in S_ALL if c.startswith("s_pace_") or c in ("s_rpci_pred",)],
    "m_(展開マッチ)": [c for c in S_ALL if c.startswith("m_")],
    "P4(上がり)": [c for c in S_ALL if c.startswith("s_l3f")],
    "P5・P6(時計・着順)": [c for c in S_ALL if c in ("s_time_pred", "s_time_gap", "s_rank_pred")],
}
F2_COLS = [c for c in BASE_FEATS if c.startswith("f2_")]
specs = [ab.Spec("base(2022〜)", BASE_FEATS, min_train_year=2022)]
specs += [ab.Spec(f"+{k}", BASE_FEATS + v, min_train_year=2022) for k, v in FAM_S.items() if v]
P4C = FAM_S["P4(上がり)"]
P56C = FAM_S["P5・P6(時計・着順)"]
specs += [ab.Spec("+P4+P5・P6", BASE_FEATS + P4C + P56C, min_train_year=2022),
          ab.Spec("+全部", BASE_FEATS + S_ALL, min_train_year=2022),
          ab.Spec("全部 − F2(簡易版)を置き換え", [c for c in BASE_FEATS if c not in F2_COLS] + S_ALL, min_train_year=2022)]
# 学習を 2020 年から使い、前段が無い 2020〜2021 年の行は欠損のままにする案（NaN 許容）。base も 2020 年から
specs += [ab.Spec("base(2020〜)", BASE_FEATS),
          ab.Spec("NaN許容 +P4", BASE_FEATS + P4C),
          ab.Spec("NaN許容 +P5・P6", BASE_FEATS + P56C),
          ab.Spec("NaN許容 +P4+P5・P6", BASE_FEATS + P4C + P56C),
          ab.Spec("NaN許容 +全部", BASE_FEATS + S_ALL)]
FEAT_HASH = hashlib.md5(pd.util.hash_pandas_object(FEATS.iloc[::40], index=False).to_numpy().tobytes() + "|".join(S_ALL).encode()).hexdigest()[:8]
RES = OOF / f"downstream_results_{FEAT_HASH}.csv"   # 前段の特徴量の中身が変わったら別ファイル（古い結果を使わない）
print("前段特徴量のハッシュ:", FEAT_HASH)
ROWS = pd.read_csv(RES).to_dict("records") if RES.exists() else []
for sp in specs:
    if any(r["key"] == sp.key() for r in ROWS):
        continue
    t0 = time.time()
    rows = ab.evaluate(df, folds, sp, SEEDS3, THREADS, targets=("win",))
    ROWS += rows
    pd.DataFrame(ROWS).to_csv(RES, index=False)
    print(f"  {sp.name:34s} {np.mean([r['logloss'] for r in rows]):.5f} ({time.time() - t0:.0f}s, {len(sp.feats)}列)")
R = pd.DataFrame(ROWS)
S = ab.summarize(R)
display(S.round(5))
names22 = [i for i in S.index if not i.startswith("NaN許容") and i != "base(2020〜)"]
CMP = ab.compare_to_base(S.loc[names22], "base(2022〜)")
print("【学習を 2022 年以降に揃える案】 base(2022〜) との差（負なら改善）"); display(CMP.round(5))
# NaN 許容案は valid 2023・2024 だけで base(2020〜) と比べる（valid 2022 では前段の特徴量が学習に一度も現れないため）
namesnan = ["base(2020〜)"] + [i for i in S.index if i.startswith("NaN許容")]
S_nan = S.loc[namesnan, ["valid_2023", "valid_2024", "seed_sd"]]
CMP_NAN = ab.compare_to_base(S_nan, "base(2020〜)")
print("【学習は 2020 年から、前段が無い行は欠損】 base(2020〜) との差（valid 2023・2024）"); display(CMP_NAN.round(5))

In [ ]:
# 採用する前段の特徴量と、学習の範囲。2 つの案を同じ基準で比べ、良い方を採る
FAM_OF = {"+P4": P4C, "+P5・P6": P56C, "+P4+P5・P6": P4C + P56C, "+全部": S_ALL}
cands = []
for scope, tbl, prefix in (("2022年以降のみ学習", CMP, ""), ("2020年から学習(2020〜2021は欠損)", CMP_NAN, "NaN許容 ")):
    for nm, feats in FAM_OF.items():
        key = prefix + nm
        if key in tbl.index and bool(tbl.loc[key, "採用基準を満たす"]):
            cands.append({"範囲": scope, "候補": nm, "Δ平均": float(tbl.loc[key, "Δ平均"]), "列数": len(feats)})
cands = pd.DataFrame(cands, columns=["範囲", "候補", "Δ平均", "列数"])
print("基準を満たした候補:"); display(cands.round(5))
adopted_scope, adopted_name, adopted_s = None, None, []
if len(cands):
    # 範囲をまたいで水準をそろえる: 2022年以降のみ案は base(2022〜) との差なので、base(2020〜) 比に換算する
    lvl = {"2022年以降のみ学習": float(S.loc["base(2022〜)", ["valid_2023", "valid_2024"]].mean() - S.loc["base(2020〜)", ["valid_2023", "valid_2024"]].mean()),
           "2020年から学習(2020〜2021は欠損)": 0.0}
    cands["base(2020〜)比"] = cands["Δ平均"] + cands["範囲"].map(lvl)
    best = cands["base(2020〜)比"].min()
    near = cands[cands["base(2020〜)比"] <= best + 0.0002].sort_values(["列数", "base(2020〜)比"])   # 同等なら列数が少ない方
    pick = near.iloc[0]
    adopted_scope, adopted_name = pick["範囲"], pick["候補"]
    adopted_s = FAM_OF[adopted_name]
    print("換算後:"); display(cands.round(5))
    print(f"採用: {adopted_scope} / {adopted_name} ({len(adopted_s)}列)  base(2020〜)比 {pick['base(2020〜)比']:+.5f}")
    print(" ", adopted_s)
else:
    print("基準を満たす前段の特徴量はありませんでした")
adopt_fams = [k for k in FAM_S if f"+{k}" in CMP.index and bool(CMP.loc[f"+{k}", "採用基準を満たす"])]
all_ok = bool(CMP.loc["+全部", "採用基準を満たす"]) if "+全部" in CMP.index else False

## 8. 記録

In [ ]:
report = {
    "git_commit": GIT_COMMIT, "cache_tag": CACHE_TAG, "n_rows": int(len(df)), "base_features": len(BASE_FEATS),
    "p1_choice": P1_CHOICE, "p3_choice": P3_CHOICE, "p3_below_ln3": P3_OK, "pace_label_for_p4": PACE_LABEL,
    "p4_uncond": P4_UNCOND, "p4_cond": P4_COND, "p5_choice": P5_CHOICE, "p6_choice": P6_CHOICE,
    "stage_features": S_ALL, "adopted_families": adopt_fams, "adopted_stage_features": adopted_s, "adopted_scope": adopted_scope, "adopted_name": adopted_name,
    "all_stage_ok": all_ok, "candidates": cands.round(6).to_dict("records"), "downstream_nan_scope": CMP_NAN.round(6).reset_index().to_dict("records"),
    "p3_table": P3T.round(5).to_dict("records"), "cmp_p4": cmp4.round(5).reset_index().to_dict("records"),
    "cmp_p5": cmp5.round(5).reset_index().to_dict("records"), "cmp_p6": cmp6.round(5).reset_index().to_dict("records"),
    "downstream": CMP.round(6).reset_index().to_dict("records"),
}
(OOF / "stage_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-04-stage-models", tags={"notebook": "tp-04", "git_commit": GIT_COMMIT, "training_years": "2020-2025", "oof": "expanding"}):
    mlflow.log_params({"n_base_features": len(BASE_FEATS), "n_stage_features": len(S_ALL), "pace_label_for_p4": PACE_LABEL,
                       "p1_early": P1_CHOICE["t_pos_early"], "p1_late": P1_CHOICE["t_pos_late"], "p3_pace_bal": P3_CHOICE["pace_bal"], "p3_rpci": P3_CHOICE["rpci"]})
    mlflow.log_metrics({f"p3_ll_{r['ラベル']}_{i}": float(r["LogLoss(seed平均)"]) for i, r in P3T.iterrows()})
    mlflow.log_metrics({"all_stage_ok": float(all_ok), "n_adopted_families": float(len(adopt_fams))})
    mlflow.log_artifact(str(OOF / "stage_report.json"), "stage_meta")
    mlflow.log_artifact(str(RES), "stage_meta")
print("記録しました。次は tp-05（モデル比較）。")